<a href="https://colab.research.google.com/github/dzidz1/Freeuni_ML_Walmart_Sales_Forecasting/blob/main/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup & MLflow

In [4]:
!pip install -q mlflow dagshub lightgbm

from google.colab import userdata
import dagshub

token = userdata.get("DAGSHUB_TOKEN")
assert token, "DAGSHUB_TOKEN secret missing or notebook access not enabled"

dagshub.auth.add_app_token(token)
dagshub.init(repo_owner="adzid23", repo_name="Freeuni_ML_Walmart_Sales_Forecasting", mlflow=True)

import mlflow
print("MLflow:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "adzid23/Freeuni_ML_Walmart_Sales_Forecasting"

Repository adzid23/Freeuni_ML_Walmart_Sales_Forecasting initialized!

MLflow: https://dagshub.com/adzid23/Freeuni_ML_Walmart_Sales_Forecasting.mlflow


## 2. Raw test set

In [5]:
import os, zipfile
import numpy as np, pandas as pd

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_KEY")
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting -q

os.makedirs("walmart_data", exist_ok=True)
with zipfile.ZipFile("walmart-recruiting-store-sales-forecasting.zip") as z:
    z.extractall("walmart_data")
for f in os.listdir("walmart_data"):
    if f.endswith(".zip"):
        with zipfile.ZipFile(f"walmart_data/{f}") as z:
            z.extractall("walmart_data")

test = pd.read_csv("walmart_data/test.csv")   # deliberately no parse_dates — raw as shipped
print(test.shape)
test.head()

(115064, 4)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


## 3. Model Registry

In [7]:
LGBM_MODEL_URI = "models:/m-73236e34515444b39e1a2a651b6095dd"

reg = mlflow.register_model(LGBM_MODEL_URI, name="walmart_best_model")
print("registered:", reg.name, "| version:", reg.version)

from mlflow import MlflowClient
client = MlflowClient()
client.update_registered_model(
    name="walmart_best_model",
    description=("LightGBM 3-seed ensemble pipeline (v3 features, log1p target, "
                 "5x holiday weights). Raw test.csv in -> Weekly_Sales out; "
                 "preprocessing internal, source csvs shipped as artifacts. "
                 "Val wmae_600=2757.2, wmae_all=1189.6; "
                 "Kaggle public 2640.3 / private 2725.0."))
print("description set")

Registered model 'walmart_best_model' already exists. Creating a new version of this model...
2026/07/12 18:34:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart_best_model, version 1
Created version '1' of model 'walmart_best_model'.


registered: walmart_best_model | version: 1
description set


## 4. Load from Model Registry and predict on raw test set

In [8]:
model = mlflow.pyfunc.load_model("models:/walmart_best_model/1")

preds = model.predict(test)          # raw test.csv, no preprocessing

out = test.copy()
out["Weekly_Sales"] = preds.values
print("rows:", len(out), "| NaN preds:", out["Weekly_Sales"].isna().sum())
print(out.head(3).to_string(index=False))

# must match the scored submission: 1_1_2012-11-02 -> 39921.66
assert abs(out.iloc[0]["Weekly_Sales"] - 39921.66) < 1.0
print("\nmatches the Kaggle-scored pipeline ✓")

/tmp/ipykernel_2722/1794197277.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


rows: 115064 | NaN preds: 0
 Store  Dept       Date  IsHoliday  Weekly_Sales
     1     1 2012-11-02      False  39921.656542
     1     1 2012-11-09      False  20752.563500
     1     1 2012-11-16      False  19826.418706

matches the Kaggle-scored pipeline ✓


## 5. Kaggle submission from the Registry model

In [9]:
sub = pd.DataFrame({
    "Id": out["Store"].astype(str) + "_" + out["Dept"].astype(str) + "_" + out["Date"],
    "Weekly_Sales": out["Weekly_Sales"]})
assert len(sub) == 115064 and sub["Weekly_Sales"].isna().sum() == 0
sub.to_csv("submission_registry_lightgbm.csv", index=False)
print(sub.head(3).to_string(index=False))

!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting \
    -f submission_registry_lightgbm.csv -m "walmart_best_model v1 from Model Registry (LightGBM)"

            Id  Weekly_Sales
1_1_2012-11-02  39921.656542
1_1_2012-11-09  20752.563500
1_1_2012-11-16  19826.418706
100% 3.84M/3.84M [00:00<00:00, 8.53MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting

In [10]:
!kaggle competitions submissions -c walmart-recruiting-store-sales-forecasting | head -5

fileName                          date                        description                                           status                     publicScore  privateScore  
--------------------------------  --------------------------  ----------------------------------------------------  -------------------------  -----------  ------------  
submission_registry_lightgbm.csv  2026-07-12 18:51:04.900000  walmart_best_model v1 from Model Registry (LightGBM)  SubmissionStatus.COMPLETE  2640.28322   2725.04534    
submission_xgboost.csv            2026-07-12 17:43:27.667000  XGBoost 3-seed ensemble, v3 features, log1p           SubmissionStatus.COMPLETE  2707.44741   2796.79012    
